# LeetCode 2073 — Time Needed to Buy Tickets bằng Python

Link đề: https://leetcode.com/problems/time-needed-to-buy-tickets/ · Độ khó: **Easy** · Chủ đề: Array, Queue, Simulation

```python
class Solution:
    def timeRequiredToBuy(self, tickets: List[int], k: int) -> int:

```

**Mục tiêu học**

1. Mô phỏng **hàng đợi (queue)** đúng như đề bằng `collections.deque` (`popleft`, `append`) — hiểu FIFO.
2. Biết vì sao `list.pop(0)` chậm và `deque` nhanh.
3. Từ mô phỏng, rút ra **công thức đếm trực tiếp** `O(n)`: mỗi người trước/sau `k` đóng góp bao nhiêu giây (dùng `min`).
4. Chứng minh công thức bằng lập luận "theo vòng".
5. Nộp `class Solution`, đối chiếu mô phỏng và công thức trên nhiều input.

## 1. Đề bài yêu cầu gì?

`n` người xếp hàng mua vé; người 0 đứng đầu. `tickets[i]` = số vé người `i` muốn mua.

- Mỗi lượt, người đứng đầu mua **1 vé** trong **1 giây**.
- Nếu còn cần vé, họ quay về **cuối hàng** (tức thì); nếu đã đủ, họ **rời hàng**.

Trả về **thời điểm** người ban đầu ở vị trí `k` mua xong vé.

| Ví dụ | tickets | k | Output |
|---|---|---|---|
| 1 | `[2,3,2]` | 2 | `6` |
| 2 | `[5,1,1,1]` | 0 | `8` |

Constraints: `1 <= n <= 100`, `1 <= tickets[i] <= 100`, `0 <= k < n`.

Diễn biến ví dụ 1 (người `k` đánh dấu `*`): `[2,3,2*]` → 1s `[3,2*,1]` → 2s `[2*,1,2]` → 3s `[1,2,1*]` → 4s `[2,1*]` (người đầu rời) → 5s `[1*,1]` → 6s người `k` mua vé cuối → **6**.

In [1]:
# Tổng số vé của mọi người là giới hạn trên của thời gian
tickets, k = [2, 3, 2], 2
print("Tổng vé:", sum(tickets), "| đáp án 6 < 7 vì người 1 còn 1 vé chưa mua khi người k xong")

Tổng vé: 7 | đáp án 6 < 7 vì người 1 còn 1 vé chưa mua khi người k xong


## 2. Nền tảng Python

### 2.1. Hàng đợi (queue) và FIFO

**Queue** là cấu trúc "**vào trước ra trước**" (FIFO — First In, First Out): lấy ra ở **đầu**, thêm vào ở **cuối** — đúng như xếp hàng mua vé.

### 2.2. `list.pop(0)` chậm, `deque.popleft()` nhanh

- `lst.pop(0)` xóa phần tử đầu list → mọi phần tử còn lại phải **dịch trái** một ô → `O(n)`.
- `collections.deque` (double-ended queue) thêm/bớt ở **hai đầu** đều `O(1)`.

In [2]:
from collections import deque
q = deque([10, 20, 30])
front = q.popleft()          # lấy ra ở đầu
q.append(front)              # đưa về cuối
print(front, q)              # 10 deque([20, 30, 10])
q.appendleft(5)
print(q, q[0], len(q))       # truy cập q[0] (đầu hàng) và độ dài

10 deque([20, 30, 10])
deque([5, 20, 30, 10]) 5 4


### 2.3. Lưu cặp `(chỉ số gốc, số vé còn lại)` trong hàng

Khi người di chuyển, vị trí trong hàng thay đổi; để biết ai là người `k` ban đầu, ta lưu **chỉ số gốc** cùng số vé. Một **tuple** `(i, t)` gói hai giá trị; "tháo" tuple bằng `i, t = q.popleft()`.

In [3]:
tickets = [2, 3, 2]
q = deque((i, t) for i, t in enumerate(tickets))
print(q)
i, t = q.popleft()
print("người gốc", i, "còn", t, "vé")

deque([(0, 2), (1, 3), (2, 2)])
người gốc 0 còn 2 vé


### 2.4. `min(a, b)` và `range` với điều kiện theo chỉ số

`min(a, b)` trả số nhỏ hơn. Dùng nhiều ở cách công thức: "người `i` mua tối đa bao nhiêu vé trước khi người `k` xong?" = `min(tickets[i], giới_hạn)`.

In [4]:
print(min(3, 2), min(1, 5), min(4, 4))
for i in range(4):
    print(i, "trước hoặc bằng k=2" if i <= 2 else "sau k=2")

2 1 4
0 trước hoặc bằng k=2
1 trước hoặc bằng k=2
2 trước hoặc bằng k=2
3 sau k=2


## 3. Cách 1 — Mô phỏng bằng `deque` (đúng nghĩa đen của đề)

Mỗi vòng lặp = 1 giây:

1. `i, t = q.popleft()`; `t -= 1`; `time += 1`.
2. Nếu `t == 0` và `i == k` → trả `time`.
3. Nếu `t > 0` → `q.append((i, t))` (về cuối hàng); nếu `t == 0` thì rời hàng (không append).

In [5]:
def time_simulate_deque(tickets, k):
    q = deque(enumerate(tickets))       # các cặp (i, tickets[i])
    time = 0
    while q:
        i, t = q.popleft()
        t -= 1
        time += 1
        if t == 0:
            if i == k:
                return time
        else:
            q.append((i, t))
    return time                          # không tới đây với input hợp lệ

print(time_simulate_deque([2, 3, 2], 2))     # 6
print(time_simulate_deque([5, 1, 1, 1], 0))  # 8

6
8


`while q:` — deque rỗng là `False`, nên vòng lặp chạy khi còn người trong hàng. `deque(enumerate(tickets))` tạo hàng từ các cặp `(i, tickets[i])`.

### TRACE mô phỏng ví dụ 1

In [6]:
def trace_simulate(tickets, k):
    q = deque(enumerate(tickets))
    time = 0
    print(f"t=0s: hàng = {[f'{t}*' if i == k else t for i, t in q]}")
    while q:
        i, t = q.popleft()
        t -= 1
        time += 1
        if t > 0:
            q.append((i, t))
        state = [f'{tt}*' if ii == k else tt for ii, tt in q]
        note = "rời hàng" if t == 0 else "về cuối"
        print(f"t={time}s: người {i} mua 1 vé ({note}) -> hàng = {state}")
        if t == 0 and i == k:
            print("Người k mua xong ->", time)
            return time

trace_simulate([2, 3, 2], 2)

t=0s: hàng = [2, 3, '2*']
t=1s: người 0 mua 1 vé (về cuối) -> hàng = [3, '2*', 1]
t=2s: người 1 mua 1 vé (về cuối) -> hàng = ['2*', 1, 2]
t=3s: người 2 mua 1 vé (về cuối) -> hàng = [1, 2, '1*']
t=4s: người 0 mua 1 vé (rời hàng) -> hàng = [2, '1*']
t=5s: người 1 mua 1 vé (về cuối) -> hàng = ['1*', 1]
t=6s: người 2 mua 1 vé (rời hàng) -> hàng = [1]
Người k mua xong -> 6


6

In [7]:
trace_simulate([5, 1, 1, 1], 0)

t=0s: hàng = ['5*', 1, 1, 1]
t=1s: người 0 mua 1 vé (về cuối) -> hàng = [1, 1, 1, '4*']
t=2s: người 1 mua 1 vé (rời hàng) -> hàng = [1, 1, '4*']
t=3s: người 2 mua 1 vé (rời hàng) -> hàng = [1, '4*']
t=4s: người 3 mua 1 vé (rời hàng) -> hàng = ['4*']
t=5s: người 0 mua 1 vé (về cuối) -> hàng = ['3*']
t=6s: người 0 mua 1 vé (về cuối) -> hàng = ['2*']
t=7s: người 0 mua 1 vé (về cuối) -> hàng = ['1*']
t=8s: người 0 mua 1 vé (rời hàng) -> hàng = []
Người k mua xong -> 8


8

### Chi phí của mô phỏng

Mỗi giây là một vòng lặp, và tổng thời gian ≤ `sum(tickets)` ≤ `100 × 100 = 10⁴`. Với constraints này mô phỏng chạy rất nhanh. Nhưng nếu `tickets[i]` tới `10⁹`, mô phỏng từng giây là bất khả thi.

## 4. Cách 2 — Mô phỏng không cần deque: quét vòng tròn trên mảng

Thay vì xoay hàng, giữ nguyên mảng và quét `i = 0, 1, ..., n-1, 0, 1, ...`. Người có `tickets[i] == 0` coi như đã rời hàng → bỏ qua. Thứ tự phục vụ giống hệt hàng đợi vì người quay về cuối hàng sẽ được phục vụ lại đúng sau những người đứng sau họ.

In [8]:
def time_simulate_array(tickets, k):
    t = list(tickets)                    # copy để không sửa input
    time = 0
    while True:
        for i in range(len(t)):
            if t[i] == 0:
                continue                 # đã rời hàng
            t[i] -= 1
            time += 1
            if i == k and t[i] == 0:
                return time

print(time_simulate_array([2, 3, 2], 2), time_simulate_array([5, 1, 1, 1], 0))

6 8


## 5. Nút thắt và insight — đếm theo "vòng"

Nhìn mô phỏng theo **vòng**: mỗi vòng, mỗi người còn trong hàng mua đúng 1 vé, theo thứ tự chỉ số gốc. Người `k` cần `T = tickets[k]` vòng. Hỏi: tới lúc người `k` mua vé thứ `T`, **mỗi người khác đã mua bao nhiêu vé?**

- Người `i` **đứng trước hoặc chính là** `k` (`i <= k`): trong mỗi vòng 1..T, họ được phục vụ **trước** `k`. Họ mua tối đa `T` vé, nhưng không quá số vé họ cần → `min(tickets[i], T)`.
- Người `i` **đứng sau** `k` (`i > k`): ở vòng thứ `T`, người `k` xong **trước khi** tới lượt họ → họ chỉ kịp mua trong `T − 1` vòng đầu → `min(tickets[i], T − 1)`.

Mỗi vé = 1 giây, nên **đáp án = tổng các đóng góp đó**.

In [9]:
tickets, k = [2, 3, 2], 2
T = tickets[k]
for i, t in enumerate(tickets):
    contrib = min(t, T) if i <= k else min(t, T - 1)
    print(f"người {i}: cần {t}, {'trước/bằng' if i <= k else 'sau'} k -> mua {contrib} vé trước khi k xong")

người 0: cần 2, trước/bằng k -> mua 2 vé trước khi k xong
người 1: cần 3, trước/bằng k -> mua 2 vé trước khi k xong
người 2: cần 2, trước/bằng k -> mua 2 vé trước khi k xong


## 6. Cách 3 (lời giải chính) — Công thức `O(n)`

```python
T = tickets[k]
sum(min(t, T) if i <= k else min(t, T - 1) for i, t in enumerate(tickets))
```

In [10]:
def time_formula(tickets, k):
    T = tickets[k]
    total = 0
    for i, t in enumerate(tickets):
        if i <= k:
            total += min(t, T)
        else:
            total += min(t, T - 1)
    return total

print(time_formula([2, 3, 2], 2))       # 6
print(time_formula([5, 1, 1, 1], 0))    # 8

6
8


## 7. TRACE công thức — bảng đóng góp

Ví dụ 2: `tickets = [5,1,1,1]`, `k = 0`, `T = 5`.

| i | tickets[i] | vị trí so với k | giới hạn | đóng góp | tổng |
|---|---|---|---|---|---|
| 0 | 5 | = k | T = 5 | 5 | 5 |
| 1 | 1 | sau | T−1 = 4 | 1 | 6 |
| 2 | 1 | sau | 4 | 1 | 7 |
| 3 | 1 | sau | 4 | 1 | 8 |

Ví dụ 1: `[2,3,2]`, `k = 2`, `T = 2`: `min(2,2) + min(3,2) + min(2,2) = 2 + 2 + 2 = 6` (không có ai sau `k`).

In [11]:
def trace_formula(tickets, k):
    T = tickets[k]
    total = 0
    print(f"T = tickets[{k}] = {T}")
    for i, t in enumerate(tickets):
        cap = T if i <= k else T - 1
        add = min(t, cap)
        total += add
        pos = "= k" if i == k else ("trước" if i < k else "sau")
        print(f"i={i}: tickets={t}, {pos:<5} giới hạn={cap} -> cộng {add}, tổng={total}")
    return total

trace_formula([5, 1, 1, 1], 0)
print()
trace_formula([3, 1, 4, 1, 5, 9], 2)

T = tickets[0] = 5
i=0: tickets=5, = k   giới hạn=5 -> cộng 5, tổng=5
i=1: tickets=1, sau   giới hạn=4 -> cộng 1, tổng=6
i=2: tickets=1, sau   giới hạn=4 -> cộng 1, tổng=7
i=3: tickets=1, sau   giới hạn=4 -> cộng 1, tổng=8

T = tickets[2] = 4
i=0: tickets=3, trước giới hạn=4 -> cộng 3, tổng=3
i=1: tickets=1, trước giới hạn=4 -> cộng 1, tổng=4
i=2: tickets=4, = k   giới hạn=4 -> cộng 4, tổng=8
i=3: tickets=1, sau   giới hạn=3 -> cộng 1, tổng=9
i=4: tickets=5, sau   giới hạn=3 -> cộng 3, tổng=12
i=5: tickets=9, sau   giới hạn=3 -> cộng 3, tổng=15


15

Đối chiếu ví dụ thứ hai bằng mô phỏng để tin công thức:

In [12]:
print(time_simulate_deque([3, 1, 4, 1, 5, 9], 2))

15


## 8. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def timeRequiredToBuy(self, tickets: List[int], k: int) -> int:
        T = tickets[k]
        return sum(min(t, T) if i <= k else min(t, T - 1)
                   for i, t in enumerate(tickets))

print(Solution().timeRequiredToBuy([2, 3, 2], 2))

6


Bản mô phỏng bằng deque cũng nộp được (chạy ≤ 10⁴ bước):

In [14]:
class SolutionSimulate:
    def timeRequiredToBuy(self, tickets: List[int], k: int) -> int:
        q = deque(enumerate(tickets))
        time = 0
        while q:
            i, t = q.popleft()
            time += 1
            if t > 1:
                q.append((i, t - 1))
            elif i == k:
                return time
        return time

print(SolutionSimulate().timeRequiredToBuy([5, 1, 1, 1], 0))

8


## 9. Vì sao công thức đúng?

**Quan sát 1 — thứ tự phục vụ theo vòng.** Hàng ban đầu theo chỉ số `0..n−1`. Ai mua xong 1 vé mà còn thiếu thì về cuối — **sau** tất cả những người còn lại. Do đó việc phục vụ diễn ra theo **vòng**: vòng `r` phục vụ (theo thứ tự chỉ số tăng) mọi người còn cần vé sau `r − 1` vòng. Người `i` có mặt ở vòng `r` ⇔ `tickets[i] >= r`.

**Quan sát 2 — thời điểm người k xong.** Người `k` mua vé cuối ở vòng `T = tickets[k]`, ngay khi tới lượt họ. Các giây đã trôi qua = số vé đã bán tính đến lúc đó, gồm:

- Vòng `1..T−1` đầy đủ: người `i` mua ở vòng `r` nếu `tickets[i] >= r` → góp `min(tickets[i], T − 1)`.
- Vòng `T` cho tới **hết lượt của `k`**: chỉ những người `i <= k` có `tickets[i] >= T` → góp thêm 1.

Với `i <= k`: `min(t, T−1) + [t >= T] = min(t, T)`. Với `i > k`: `min(t, T−1)`. Cộng lại đúng công thức. ∎

In [15]:
# Kiểm tra đẳng thức min(t, T-1) + (t >= T) == min(t, T) trên nhiều giá trị
assert all(min(t, T - 1) + (t >= T) == min(t, T) for t in range(1, 30) for T in range(1, 30))
print("Đẳng thức đúng với mọi t, T trong 1..29")

Đẳng thức đúng với mọi t, T trong 1..29


## 10. Độ phức tạp

Gọi `n = len(tickets)`, `S = sum(tickets)` (≤ 10⁴).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng deque | `O(S)` (mỗi giây `O(1)`) | `O(n)` | đúng nghĩa đen, dễ kiểm chứng |
| 1'. Mô phỏng bằng `list.pop(0)` | `O(S · n)` | `O(n)` | chậm vì dịch phần tử |
| 2. Quét vòng trên mảng | `O(T · n)` với `T = tickets[k]` | `O(n)` | không cần deque |
| 3. Công thức | `O(n)` | `O(1)` | tối ưu, không phụ thuộc độ lớn vé |

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS ({actual})")

cases = [
    ("ví dụ 1", [2, 3, 2], 2, 6),
    ("ví dụ 2", [5, 1, 1, 1], 0, 8),
    ("n = 1", [7], 0, 7),
    ("k đầu hàng, 1 vé", [1, 100, 100], 0, 1),
    ("k cuối hàng", [3, 3, 3], 2, 9),
    ("k giữa, người sau ít vé", [2, 4, 1, 1], 1, 8),
    ("mọi người 1 vé", [1, 1, 1, 1], 2, 3),
    ("người sau nhiều vé", [1, 2, 9, 9], 1, 5),
]
fs = [time_simulate_deque, time_simulate_array, time_formula,
      Solution().timeRequiredToBuy, SolutionSimulate().timeRequiredToBuy]
for name, t, k, exp in cases:
    for f in fs:
        assert f(list(t), k) == exp, (f, t, k, f(list(t), k))
    check(name, Solution().timeRequiredToBuy(t, k), exp)

ví dụ 1                            PASS (6)
ví dụ 2                            PASS (8)
n = 1                              PASS (7)
k đầu hàng, 1 vé                   PASS (1)
k cuối hàng                        PASS (9)
k giữa, người sau ít vé            PASS (8)
mọi người 1 vé                     PASS (3)
người sau nhiều vé                 PASS (5)


In [17]:
import random
random.seed(2073)
for _ in range(2000):
    n = random.randint(1, 8)
    t = [random.randint(1, 6) for _ in range(n)]
    k = random.randrange(n)
    answers = {f(list(t), k) for f in fs}
    assert len(answers) == 1, (t, k, [f(list(t), k) for f in fs])
print("2000 bộ ngẫu nhiên: mô phỏng và công thức khớp nhau — PASS")

2000 bộ ngẫu nhiên: mô phỏng và công thức khớp nhau — PASS


Tự kiểm một ca bằng tay trước khi tin test: `[2,4,1,1]`, `k = 1`, `T = 4` → `min(2,4) + min(4,4) + min(1,3) + min(1,3) = 2 + 4 + 1 + 1 = 8`. Người 0 mua hết 2 vé, người 2 và 3 mỗi người 1 vé, người 1 mua 4 vé → tổng 8 vé = 8 giây. Ca `[1,2,9,9]`, `k = 1`, `T = 2`: `1 + 2 + min(9,1) + min(9,1) = 5` — hai người sau `k` chỉ kịp mua **1** vé (vòng 1), ở vòng 2 người `k` xong trước họ.

Trace mô phỏng để thấy tận mắt:

In [18]:
trace_simulate([1, 2, 9, 9], 1)
print("Công thức:", time_formula([1, 2, 9, 9], 1))

t=0s: hàng = [1, '2*', 9, 9]
t=1s: người 0 mua 1 vé (rời hàng) -> hàng = ['2*', 9, 9]
t=2s: người 1 mua 1 vé (về cuối) -> hàng = [9, 9, '1*']
t=3s: người 2 mua 1 vé (về cuối) -> hàng = [9, '1*', 8]
t=4s: người 3 mua 1 vé (về cuối) -> hàng = ['1*', 8, 8]
t=5s: người 1 mua 1 vé (rời hàng) -> hàng = [8, 8]
Người k mua xong -> 5
Công thức: 5


Bài học: khi tính tay và code lệch nhau, **đừng sửa expected cho khớp code** — hãy trace để tìm xem bên nào sai.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng cùng một giới hạn `T` cho mọi người (quên phân biệt trước/sau `k`)

In [19]:
def wrong_same_cap(tickets, k):
    T = tickets[k]
    return sum(min(t, T) for t in tickets)

print("Sai :", wrong_same_cap([5, 1, 1, 1], 0), "| đúng:", time_formula([5, 1, 1, 1], 0))   # trùng tình cờ
print("Sai :", wrong_same_cap([2, 3, 2], 0), "| đúng:", time_formula([2, 3, 2], 0))         # lộ lỗi

Sai : 8 | đúng: 8
Sai : 6 | đúng: 4


Ví dụ 2 của đề **không bắt được** lỗi này (người sau chỉ có 1 vé). Phải có test người **sau** `k` có ≥ `T` vé.

### Lỗi 2: Đếm thời gian khi **bất kỳ ai** mua xong, không phải riêng người `k`

Quên điều kiện `i == k` → trả về thời điểm người đầu tiên rời hàng.

### Lỗi 3: Theo dõi người `k` bằng **vị trí trong hàng** thay vì chỉ số gốc

Sau khi hàng xoay, vị trí `k` chứa người khác. Phải lưu chỉ số gốc trong hàng (cặp `(i, t)`).

In [20]:
def wrong_track_position(tickets, k):
    q = deque(tickets)               # mất thông tin "ai là ai"
    time = 0
    while q:
        t = q.popleft() - 1
        time += 1
        if t > 0:
            q.append(t)
        elif len(q) == k:            # đoán mò theo vị trí -> sai
            return time
    return time

print("Sai :", wrong_track_position([2, 3, 2], 2), "| đúng:", time_simulate_deque([2, 3, 2], 2))

Sai : 4 | đúng: 6


### Lỗi 4: Sửa trực tiếp `tickets` trong mô phỏng rồi dùng lại

`time_simulate_array` gốc sửa mảng; nếu không `list(tickets)` copy, lần gọi thứ hai với cùng biến sẽ sai (mảng đã bị trừ). Đây là lý do test dùng `list(t)`.

In [21]:
def sim_mutating(t, k):
    time = 0
    while True:
        for i in range(len(t)):
            if t[i] == 0:
                continue
            t[i] -= 1
            time += 1
            if i == k and t[i] == 0:
                return time

data = [2, 3, 2]
print(sim_mutating(data, 2), data)        # data đã bị sửa thành [0, 1, 0]

6 [0, 1, 0]


## 13. Mẹo nhận dạng pattern

- Đề mô tả **xếp hàng / quay vòng / về cuối hàng** → mô phỏng bằng `deque` (`popleft` + `append`).
- Khi mô phỏng theo từng đơn vị thời gian có thể quá lớn → tìm **cấu trúc theo vòng**: mỗi vòng ai cũng đi 1 bước → đóng góp của mỗi phần tử là `min(nhu cầu, số vòng)`.
- Phân biệt phần tử **đứng trước** và **đứng sau** mục tiêu trong vòng cuối → `T` và `T − 1`.
- Bài cùng họ: 1700 Number of Students Unable to Eat Lunch (queue), 950 Reveal Cards In Increasing Order (deque), 933 Number of Recent Calls, 1823 Find the Winner of the Circular Game (Josephus).

## 14. Biến thể

1. **Thời điểm mọi người xong:** đáp án cho từng `k` (gọi công thức `n` lần → `O(n²)`; có cách `O(n log n)` bằng sắp xếp).
2. **Mỗi lượt mua tối đa `m` vé** (thay vì 1): người `i` cần `ceil(tickets[i] / m)` lượt, nhưng thời gian = số **vé**, cần cẩn thận ở lượt cuối.
3. **Thứ tự rời hàng:** trả về danh sách chỉ số theo thứ tự mua xong.

In [22]:
def all_finish_times(tickets):
    return [time_formula(tickets, k) for k in range(len(tickets))]

def leave_order(tickets):
    q = deque(enumerate(tickets))
    order = []
    while q:
        i, t = q.popleft()
        if t > 1:
            q.append((i, t - 1))
        else:
            order.append(i)
    return order

print(all_finish_times([2, 3, 2]))     # [4, 7, 6]
print(leave_order([2, 3, 2]))          # [0, 2, 1]

[4, 7, 6]
[0, 2, 1]


## 15. Bài tập tự luyện

**Bài 1 — Tính tay:** `tickets = [4, 2, 3, 1]`, `k = 2`. Tính đóng góp từng người theo công thức và đáp án.

**Bài 2 — Đoán output:** sau `q = deque([1, 2, 3]); q.append(q.popleft()); q.append(q.popleft())`, `q` là gì?

**Bài 3 — TODO:** `exercise_simulate(tickets, k)` — mô phỏng bằng `deque` với cặp `(i, t)`.

**Bài 4 — TODO:** `exercise_formula(tickets, k)` — cài công thức `O(n)`.

**Bài 5 — TODO biến thể:** `leave_order_exercise(tickets)` trả thứ tự chỉ số rời hàng **không mô phỏng từng giây** (gợi ý: sắp xếp theo `(tickets[i], i)`).

**Bài 6 — Mini challenge:** `time_with_m(tickets, k, m)` — mỗi lượt ở đầu hàng, người đó mua `min(m, vé còn lại)` vé, **mỗi vé 1 giây**. Viết bằng mô phỏng deque và kiểm tra với `m = 1` ra cùng kết quả bài gốc.

In [23]:
def exercise_simulate(tickets, k):
    # TODO: q = deque(enumerate(tickets)); vòng while; popleft; time += 1; ...
    pass

# assert exercise_simulate([2,3,2], 2) == 6

In [24]:
def exercise_formula(tickets, k):
    # TODO: T = tickets[k]; cộng min(t, T) hoặc min(t, T-1)
    pass

# assert exercise_formula([5,1,1,1], 0) == 8

In [25]:
def leave_order_exercise(tickets):
    # TODO: người có ít vé hơn rời trước; cùng số vé thì chỉ số nhỏ rời trước
    pass

# assert leave_order_exercise([2, 3, 2]) == [0, 2, 1]

In [26]:
def time_with_m(tickets, k, m):
    # TODO
    pass

# assert time_with_m([2,3,2], 2, 1) == 6
# assert time_with_m([2,3,2], 2, 2) == 6

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | T = 3 | người 0,1 trước k; người 3 sau k | `3 + 2 + 3 + 1 = 9` |
| 2 | mỗi lần đưa đầu về cuối | sau 1 lần: [2,3,1] | `deque([3, 1, 2])` |
| 3 | mục 3 | lưu `(i, t)` | `if t > 1: append((i, t-1)) elif i == k: return time` |
| 4 | mục 6 | `i <= k` dùng `T`, còn lại `T - 1` | `sum(...)` |
| 5 | rời hàng ở vòng `tickets[i]` | trong cùng vòng, thứ tự theo chỉ số | `sorted(range(n), key=lambda i: (tickets[i], i))` |
| 6 | mỗi lượt cộng `min(m, t)` giây | người k xong khi hết vé | `time += buy; t -= buy` |

## 17. Đáp án tham khảo

**Bài 1:** `T = 3`: người 0 `min(4,3)=3`, người 1 `min(2,3)=2`, người 2 `min(3,3)=3`, người 3 (sau) `min(1,2)=1` → **9**.

**Bài 2:** `deque([3, 1, 2])`.

In [27]:
trace_formula([4, 2, 3, 1], 2)
q = deque([1, 2, 3]); q.append(q.popleft()); q.append(q.popleft())
print(q)

T = tickets[2] = 3
i=0: tickets=4, trước giới hạn=3 -> cộng 3, tổng=3
i=1: tickets=2, trước giới hạn=3 -> cộng 2, tổng=5
i=2: tickets=3, = k   giới hạn=3 -> cộng 3, tổng=8
i=3: tickets=1, sau   giới hạn=2 -> cộng 1, tổng=9
deque([3, 1, 2])


In [28]:
def exercise_simulate_answer(tickets, k):
    q = deque(enumerate(tickets))
    time = 0
    while q:
        i, t = q.popleft()
        time += 1
        if t > 1:
            q.append((i, t - 1))
        elif i == k:
            return time
    return time

def exercise_formula_answer(tickets, k):
    T = tickets[k]
    return sum(min(t, T) if i <= k else min(t, T - 1) for i, t in enumerate(tickets))

def leave_order_exercise_answer(tickets):
    return sorted(range(len(tickets)), key=lambda i: (tickets[i], i))

def time_with_m_answer(tickets, k, m):
    q = deque(enumerate(tickets))
    time = 0
    while q:
        i, t = q.popleft()
        buy = min(m, t)
        time += buy
        t -= buy
        if t > 0:
            q.append((i, t))
        elif i == k:
            return time
    return time

assert exercise_simulate_answer([2, 3, 2], 2) == 6
assert exercise_formula_answer([5, 1, 1, 1], 0) == 8
assert leave_order_exercise_answer([2, 3, 2]) == [0, 2, 1]
assert time_with_m_answer([2, 3, 2], 2, 1) == 6
assert time_with_m_answer([2, 3, 2], 2, 2) == 6
random.seed(9)
for _ in range(500):
    n = random.randint(1, 7)
    t = [random.randint(1, 6) for _ in range(n)]
    k = random.randrange(n)
    assert exercise_simulate_answer(t, k) == exercise_formula_answer(t, k) == time_with_m_answer(t, k, 1)
    assert leave_order_exercise_answer(t) == leave_order(t)
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


Bài 6 với `m = 2`, `[2,3,2]`, `k = 2`: người 0 mua 2 (2s, rời), người 1 mua 2 (4s, còn 1), người 2 mua 2 (6s, xong) → 6.

## 18. Tổng kết — template + checklist

**Template mô phỏng hàng đợi:**

```python
q = deque(enumerate(items))      # (chỉ số gốc, trạng thái)
while q:
    i, s = q.popleft()
    ... xử lý 1 bước ...
    if còn việc: q.append((i, s_mới))
    elif i == target: return kết quả
```

**Công thức bài này:** `T = tickets[k]`; `sum(min(t, T) if i <= k else min(t, T - 1))`.

**Checklist submit**

- [ ] Người **trước hoặc bằng** `k` dùng `T`, người **sau** dùng `T − 1`.
- [ ] Có test người sau `k` có nhiều vé (bắt lỗi dùng chung `T`).
- [ ] Nếu mô phỏng: theo dõi **chỉ số gốc**, dùng `deque` (không `pop(0)`).
- [ ] Không sửa input nếu còn dùng lại.
- [ ] Tên method `timeRequiredToBuy`, trả `int`.